# Redis Memory — Cross-Session Agent Memory with LangGraph
## Conversation History That Survives Process Restarts
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/82-redis-memory/redis_memory_workbook.ipynb)

Most LLM agents forget everything the moment the Python process exits. **Redis-backed memory** solves this by persisting conversation history in an external key-value store that survives restarts, container redeploys, and new sessions.

This workshop builds a LangGraph agent that reads and writes its own conversation log from Redis. By the end you will understand *how* list-based Redis history works, *why* the load → respond → save pattern is safe under concurrent sessions, and *how* to inspect memory state directly.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Redis lists as conversation logs |
| 2 | **Nodes** — `load_history`, `respond`, `save_history` |
| 3 | **Graph** — Wiring the three nodes with LangGraph |
| 4 | **Seed and Run** — Pre-populate history and run queries |
| 5 | **Inspect Redis** — Read raw list contents directly |
| ★ | **Exercises** — Multi-user, TTL, and fresh-start patterns |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below handles everything)
- `OPENAI_API_KEY` in `.env` or Colab Secrets
- Redis running locally (`redis-server`) — **Colab cell installs + starts Redis automatically**

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    # Install Redis server binary (available via apt in Colab)
    subprocess.run(["apt-get", "install", "-y", "-q", "redis-server"], check=True)
    # Start Redis as a background daemon — Colab VMs don't have it running by default
    subprocess.Popen(["redis-server", "--daemonize", "yes"])
    import time; time.sleep(1)  # give Redis a moment to bind to :6379
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "redis", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete — Redis server started.")
else:
    print("Local environment — skipping install.")
    print("Make sure Redis is running: redis-server")

In [ ]:
import os

# Colab stores secrets in userdata; local dev uses a .env file
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print("API key loaded." if key else "WARNING: OPENAI_API_KEY not set.")

---
## Part 1 — The Core Concept

### Why Redis for memory?

In-process lists (`state["history"]`) vanish when your program exits. Redis is a **persistent, networked data structure server**. When you store a conversation turn in Redis, it survives:
- Process restarts
- Container redeploys
- New user sessions hours or days later

### The Redis list pattern

Redis lists are ordered sequences of strings. We use one list per user, keyed by `history:{user_id}`:

```
history:user_42  →  [ '{"role":"user","content":"Hi"}',
                       '{"role":"assistant","content":"Hello!"}',
                       ... ]
```

- **`RPUSH`** appends a new turn to the right (end) of the list
- **`LRANGE key 0 -1`** retrieves the entire list in order
- Each entry is a JSON-serialized `{role, content}` dict

This gives us a **chronological, append-only conversation log** that is O(1) to write and O(N) to read — exactly what we need.

### The LangGraph graph

```
START
  │
  ▼
[load_history]   ← fetch all prior turns from Redis into state
  │
  ▼
[respond]        ← call gpt-4o-mini with full history as context
  │
  ▼
[save_history]   ← append this user turn + assistant answer to Redis
  │
  ▼
 END
```

The key insight: **state flows through the graph, but Redis is the durable store**. The graph node `load_history` bridges them — it reads from Redis and injects the result into the LangGraph state dict so `respond` can use it.

In [ ]:
import json
import redis

# Connect to local Redis — decode_responses=True means we get strings back, not bytes
# decode_responses=True avoids manual .decode('utf-8') on every LRANGE result — always set this for string data
r = redis.Redis(host="localhost", port=6379, decode_responses=True)

# ping() raises ConnectionError if Redis isn't running — fail fast here
# ping() raises ConnectionError if Redis isn't running — fail fast here before the agent tries to use it
print("Redis connected:", r.ping())

# Quick sanity check: round-trip a value
# round-trip sanity check — confirms reads AND writes work, not just the connection
r.set("workshop:test", "hello")
val = r.get("workshop:test")
print(f"Round-trip test: {val}")
r.delete("workshop:test")

---
## Part 2 — Define State and Nodes

### The state schema

LangGraph passes a `state` dict between nodes. Each node receives the full state and returns a partial dict of the keys it updates.

Our state has four fields:
- `user_id` — which user's history key to look up in Redis
- `query` — the current user message
- `history` — the list of prior turns (loaded from Redis, not persisted in-memory)
- `answer` — the LLM's response (set by `respond`, read by `save_history`)

### Why separate `load_history` from `respond`?

Separation of concerns: `load_history` owns the Redis read, `respond` owns the LLM call. This makes each node testable in isolation and makes it easy to swap Redis for another store later without touching the LLM logic.

In [ ]:
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

# gpt-4o-mini is fast, cheap, and more than capable for conversational memory tasks
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


class RedisState(TypedDict):
    user_id: str
    query: str
    history: list[dict]  # populated at runtime from Redis, not stored between invocations
    answer: str


# ── Node 1: load_history ──────────────────────────────────────────────────────
# Reads the entire conversation log for this user from Redis.
# LRANGE key 0 -1 returns all elements in insertion order.
# Each element is a JSON string — we deserialize into a list of dicts.
def load_history(state: RedisState) -> dict:
    key = f"history:{state['user_id']}"
    raw_items = r.lrange(key, 0, -1)  # list of JSON strings, oldest first
    history = [json.loads(item) for item in raw_items]
    print(f"  [load_history] Loaded {len(history)} turns for {state['user_id']}")
    return {"history": history}


# ── Node 2: respond ───────────────────────────────────────────────────────────
# Formats history as a plain-text context block and calls the LLM.
# The prompt structure is simple and explicit: prior turns are labeled by role,
# then the current query is appended. The LLM sees the full conversation arc.
def respond(state: RedisState) -> dict:
    context_lines = [
        f"[{msg['role']}] {msg['content']}" for msg in state["history"]
    ]
    context = "\n".join(context_lines) if context_lines else "(no prior history)"
    prompt = (
        f"Prior conversation history:\n{context}\n\n"
        f"User: {state['query']}\n"
        "Answer using the history if relevant. Be concise."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return {"answer": response.content.strip()}


# ── Node 3: save_history ──────────────────────────────────────────────────────
# Appends the user turn and assistant answer as two separate list entries.
# RPUSH appends to the right (end) — preserving chronological order.
# Crucially, this runs AFTER respond, so we always save a matched pair.
def save_history(state: RedisState) -> dict:
    key = f"history:{state['user_id']}"
    r.rpush(key, json.dumps({"role": "user",      "content": state["query"]}))
    r.rpush(key, json.dumps({"role": "assistant", "content": state["answer"]}))
    total = r.llen(key)
    print(f"  [save_history] Redis list now has {total} entries")
    return {}


print("State schema and nodes defined.")

---
## Part 3 — Build the Graph

LangGraph's `StateGraph` wires nodes into a directed acyclic graph. We define:
1. Which nodes exist (`add_node`)
2. How control flows between them (`add_edge`)
3. The compiled `app` — an object we can call with `.invoke()`

The graph is **linear** here: no branches, no conditional routing. Every invocation follows exactly the same path. This is intentional — for more complex memory patterns (e.g., summarize when history exceeds N turns), you would add a conditional edge after `load_history`.

In [ ]:
# Build the StateGraph
# StateGraph enforces typed state — keys not in RedisState are silently dropped, preventing state pollution
builder = StateGraph(RedisState)

builder.add_node("load_history", load_history)
builder.add_node("respond",      respond)
builder.add_node("save_history", save_history)

# Wire edges: START → load_history → respond → save_history → END
builder.add_edge(START,          "load_history")
builder.add_edge("load_history", "respond")
builder.add_edge("respond",      "save_history")
builder.add_edge("save_history", END)

# compile() locks the topology — after this, add_node/add_edge raise errors; rebuild to change the graph
app = builder.compile()

print("Graph compiled. Node execution order:")
print("  START → load_history → respond → save_history → END")
print()

# Visualize the graph structure (works in Jupyter/Colab)
try:
    from IPython.display import Image, display
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # draw_mermaid_png requires graphviz; safe to skip if unavailable
    print("(Graph visualization unavailable — install graphviz to enable)")

---
## Part 4 — Seed Prior Context and Run Queries

### Why seed prior context?

Real agents accumulate history over many sessions — the user might have spoken to the agent yesterday, last week, or last month. To simulate this in our demo, we **pre-populate Redis** with a set of prior conversation turns before running any queries.

This is exactly what a production system does on first launch: migrate existing conversation records into Redis, or let the agent build history naturally over many sessions.

The three queries then demonstrate that the agent can recall facts from those pre-seeded turns — **without those turns being passed in the current invocation**. The information comes from Redis, not from the Python session.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
USER_ID = "user_42"

# These are the "prior session" messages we inject into Redis to simulate
# an existing conversation history the agent should remember.
SESSION_MESSAGES = [
    {"role": "user",      "content": "My name is Alex and I love building AI agents."},
    {"role": "assistant", "content": "That's great, Alex! AI agents are fascinating."},
    {"role": "user",      "content": "I use Neovim and prefer Python for everything."},
    {"role": "assistant", "content": "Solid choices — Neovim + Python is a powerful combo."},
    {"role": "user",      "content": "This week I'm learning LangGraph and Redis memory patterns."},
    {"role": "assistant", "content": "LangGraph + Redis is a great combination for persistent agents!"},
]

# ── Seed Redis ────────────────────────────────────────────────────────────────
# Start fresh: delete any prior data for this user so the demo is reproducible
history_key = f"history:{USER_ID}"
r.delete(history_key)

# RPUSH each message — preserves chronological order (left to right = old to new)
for msg in SESSION_MESSAGES:
    r.rpush(history_key, json.dumps(msg))

seeded_count = r.llen(history_key)
print(f"Seeded Redis key '{history_key}' with {seeded_count} prior turns.")
print()

# ── Run 3 queries ─────────────────────────────────────────────────────────────
# Each query is an independent .invoke() call — no Python-level state is shared.
# The agent's memory comes entirely from Redis.
queries = [
    "What's my name and what do I love?",
    "What editor do I use?",
    "What am I learning this week?",
]

initial_state = {
    "user_id": USER_ID,
    "query": "",        # overridden per query below
    "history": [],      # always starts empty — load_history fills this from Redis
    "answer": "",
}

for i, query in enumerate(queries, 1):
    print(f"=== Query {i} ===")
    turns_before = r.llen(history_key)
    print(f"Redis has {turns_before} turns before this invocation")

    result = app.invoke({**initial_state, "query": query})

    print(f"Q: {query}")
    print(f"A: {result['answer']}")
    print()

---
## Part 5 — Inspect Redis Directly

One of the advantages of Redis as a memory store is **inspectability**. Unlike opaque vector databases or embedded SQLite files, Redis data is easy to read with standard tools.

Here we use the Python client directly to read the raw list contents — exactly what `load_history` sees when it runs. This lets you:
- Debug unexpected agent behavior by checking what's actually in memory
- Audit conversation history without running the full agent
- Build admin tooling that reads/edits history independently of the agent

In [ ]:
import pprint

history_key = f"history:{USER_ID}"

# LRANGE key 0 -1 — fetch all entries (index 0 to last)
raw_entries = r.lrange(history_key, 0, -1)

print(f"Key: {history_key}")
print(f"Total entries: {len(raw_entries)}")
print()

# Deserialize and display each turn
history = [json.loads(entry) for entry in raw_entries]

for idx, turn in enumerate(history):
    role    = turn["role"].upper().ljust(9)  # pad for alignment
    content = turn["content"]
    print(f"[{idx:02d}] {role} {content}")

print()
print(f"Seeded turns: {len(SESSION_MESSAGES)}")
print(f"Query turns added: {len(history) - len(SESSION_MESSAGES)} (2 per query = user + assistant)")

---
## Exercises

### Exercise 1 — New User, Fresh Session

Change `USER_ID` to `"user_99"` and run a query. The agent should have no memory of Alex — it's a completely independent conversation log.

**Why this works**: Redis key `history:user_99` doesn't exist yet, so `LRANGE` returns an empty list. The agent answers without any prior context.

### Exercise 2 — Flush and Start Fresh

Use `r.delete(f"history:{USER_ID}")` to wipe a user's history, then re-run the same queries. The agent should answer as if it has never spoken to Alex before.

**Use case**: GDPR "right to be forgotten" — deleting a Redis key is an O(1) operation.

### Exercise 3 — Add a TTL (Time to Live)

Use `r.expire(f"history:{USER_ID}", 3600)` to make the conversation expire after 1 hour. Check the remaining TTL with `r.ttl(f"history:{USER_ID}")`.

**Use case**: Session-scoped memory for customer support bots — forget the conversation after the session window closes.

In [ ]:
# ── Exercise 1 — New user, fresh session ──────────────────────────────────────
NEW_USER_ID = "user_99"
new_key = f"history:{NEW_USER_ID}"

# This key doesn't exist yet — llen returns 0
print(f"Turns for {NEW_USER_ID}: {r.llen(new_key)}")

result = app.invoke({
    "user_id": NEW_USER_ID,
    "query": "What's my name and what do I love?",
    "history": [],
    "answer": "",
})
print(f"Q: What's my name and what do I love?")
print(f"A: {result['answer']}")
print()

# ── Exercise 2 — Flush history ────────────────────────────────────────────────
print(f"Before flush: {r.llen(f'history:{USER_ID}')} turns for {USER_ID}")
r.delete(f"history:{USER_ID}")
print(f"After  flush: {r.llen(f'history:{USER_ID}')} turns for {USER_ID}")
print()

# ── Exercise 3 — Add TTL ──────────────────────────────────────────────────────
# Re-seed user_42 first so the key exists
for msg in SESSION_MESSAGES:
    r.rpush(f"history:{USER_ID}", json.dumps(msg))

r.expire(f"history:{USER_ID}", 3600)  # expire after 1 hour
ttl = r.ttl(f"history:{USER_ID}")
print(f"TTL on history:{USER_ID}: {ttl} seconds (~{ttl // 60} minutes)")
print("After TTL expires, r.lrange() will return [] and the agent starts fresh.")

---
## Part 6 — Summarization When History Is Too Long

### The context window problem

Redis can store thousands of turns — but your LLM's context window is finite. Feeding 500 turns of history into GPT-4o-mini will:
- Exceed the context window (currently ~128K tokens)
- Slow down inference significantly
- Cost more per request

### The summarization strategy

When `LLEN history:{user_id}` exceeds a threshold (e.g., 20 turns):
1. Fetch the oldest N turns from Redis
2. Ask the LLM to compress them into a summary sentence or paragraph
3. Delete those N turns from Redis
4. Prepend the summary as a special `{"role": "summary", "content": "..."}` entry

The agent node treats summary entries differently from regular `user`/`assistant` entries — it passes them as a single condensed context message rather than individual history turns.

### When to summarize

- After every `invoke()` when `llen > MAX_TURNS`
- As a background cron job (don't block user requests)
- On session start before the first turn

A sliding window of the most recent 10–15 turns plus one summary covers most conversational contexts.

In [ ]:
import json
MAX_TURNS_BEFORE_SUMMARY = 6  # low for demo; use 20-30 in production

def should_summarize(r, user_id: str) -> bool:
    return r.llen(f"history:{user_id}") > MAX_TURNS_BEFORE_SUMMARY

def summarize_oldest_turns(r, llm, user_id: str, n: int = 4) -> str:
    key     = f"history:{user_id}"
    oldest  = r.lrange(key, 0, n - 1)
    turns   = [json.loads(t) for t in oldest]

    turns_text = "\n".join(
        f"{t['role'].upper()}: {t['content']}" for t in turns
    )
    prompt = (
        f"Summarize this conversation history in 2-3 sentences, "
        f"preserving key facts the user shared:\n\n{turns_text}"
    )
    from langchain_core.messages import HumanMessage
    summary = llm.invoke([HumanMessage(content=prompt)]).content.strip()

    # atomically: remove oldest n, prepend summary
    pipe = r.pipeline()
    for _ in range(n):
        pipe.lpop(key)  # remove from left (oldest)
    pipe.lpush(key, json.dumps({"role": "summary", "content": summary}))
    pipe.execute()

    return summary

def load_history_with_summary_support(state):
    key     = f"history:{state['user_id']}"
    raw     = r.lrange(key, 0, -1)
    history = [json.loads(e) for e in raw]
    return {"history": history}

def respond_with_summary_support(state):
    from langchain_core.messages import SystemMessage, HumanMessage
    msgs = []
    for turn in state["history"]:
        if turn["role"] == "summary":
            # condensed past context — pass as system message so it doesn't inflate token count as much
            msgs.append(SystemMessage(content=f"Prior conversation summary: {turn['content']}"))
        elif turn["role"] == "user":
            msgs.append(HumanMessage(content=turn["content"]))
        else:
            from langchain_core.messages import AIMessage
            msgs.append(AIMessage(content=turn["content"]))
    msgs.append(HumanMessage(content=state["query"]))
    r_msg = llm.invoke(msgs)
    return {"response": r_msg.content.strip()}

# --- demo: fill history beyond threshold, then summarize ---
DEMO_USER = "user_summary_demo"
r.delete(f"history:{DEMO_USER}")

# fill with 8 turns to exceed MAX_TURNS_BEFORE_SUMMARY=6
demo_turns = [
    {"role": "user",      "content": "My name is Sam and I work as a data engineer."},
    {"role": "assistant", "content": "Nice to meet you, Sam!"},
    {"role": "user",      "content": "I'm building a real-time data pipeline at work."},
    {"role": "assistant", "content": "That sounds exciting! What tools are you using?"},
    {"role": "user",      "content": "Kafka, Spark, and dbt."},
    {"role": "assistant", "content": "Great stack for streaming."},
    {"role": "user",      "content": "We're migrating from Airflow to Prefect."},
    {"role": "assistant", "content": "Prefect 2.x has much nicer ergonomics."},
]
for turn in demo_turns:
    r.rpush(f"history:{DEMO_USER}", json.dumps(turn))

print(f"History length before summarization: {r.llen(f'history:{DEMO_USER}')}")
print(f"Should summarize: {should_summarize(r, DEMO_USER)}")

if should_summarize(r, DEMO_USER):
    summary = summarize_oldest_turns(r, llm, DEMO_USER, n=4)
    print(f"\nSummary generated:\n  {summary}")
    print(f"\nHistory length after summarization: {r.llen(f'history:{DEMO_USER}')}")
    print("\nRemaining entries:")
    for entry in r.lrange(f"history:{DEMO_USER}", 0, -1):
        parsed = json.loads(entry)
        print(f"  [{parsed['role']}] {parsed['content'][:80]}")

---
## Exercises (Additional)

### Exercise 4 — Summarization Threshold

Set `MAX_TURNS_BEFORE_SUMMARY = 4`. Seed a user with 6 turns and verify that `should_summarize()` returns `True`. Run the summarization and confirm the history is reduced to 3 entries (1 summary + 2 remaining turns).

### Exercise 5 — Multi-User Leaderboard

Seed three users (`user_a`, `user_b`, `user_c`) with different numbers of turns. Write a function `history_stats(r, user_ids)` that returns a dict mapping user_id → turn count. Print the results sorted by count descending.

*Hint:* Use `r.llen(f"history:{uid}")` and sort the dict.

---
## Answer Key

Attempt the exercises before reading below.

In [ ]:
# Answer 2 — Flush and Start Fresh
print(f"Turns before flush: {r.llen(f'history:{USER_ID}')}")
r.delete(f"history:{USER_ID}")
print(f"Turns after flush:  {r.llen(f'history:{USER_ID}')}")

# Re-run a query — no memory of prior sessions
result = app.invoke({
    "user_id": USER_ID,
    "query":   "What do you know about me?",
    "history": [],
    "response": "",
})
print(f"Response after flush: {result['response'][:200]}")

In [ ]:
# Answer 3 — Add a TTL (Time to Live)
TTL_SECONDS = 3600  # 1 hour

r.expire(f"history:{USER_ID}", TTL_SECONDS)
ttl = r.ttl(f"history:{USER_ID}")
print(f"TTL set to {TTL_SECONDS}s")
print(f"Remaining TTL: {ttl}s (~{ttl // 60} minutes)")

# Production note: set the TTL after every save_history call
# so it resets to full duration on each new interaction

In [ ]:
# Answer 5 — Multi-User Leaderboard
def history_stats(r, user_ids: list[str]) -> dict[str, int]:
    return {uid: r.llen(f"history:{uid}") for uid in user_ids}

# seed three users with different turn counts
test_users = ["user_alpha", "user_beta", "user_gamma"]
r.delete(*[f"history:{u}" for u in test_users])

for i, uid in enumerate(test_users):
    for j in range((i + 1) * 3):  # 3, 6, 9 turns
        r.rpush(f"history:{uid}", json.dumps({
            "role": "user" if j % 2 == 0 else "assistant",
            "content": f"Message {j} for {uid}"
        }))

stats = history_stats(r, test_users)
print("History stats (sorted by turn count):")
for uid, count in sorted(stats.items(), key=lambda x: x[1], reverse=True):
    print(f"  {uid}: {count} turns")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*